# Fixtures

Load a small synthetic sales series from CSV, split it into history and target at a
forecast origin, and make the simplest possible forecast (seasonal naive).

In [ ]:
#| default_exp fixtures

In [ ]:
#| export
import csv
from collections.abc import Sequence
from dataclasses import dataclass
from math import isfinite
from pathlib import Path

One day's value for one series.

In [ ]:
#| export
@dataclass(frozen=True)
class Observation:
    series_id: str
    day: int
    value: float

Read a CSV with exactly the columns `series_id,day,value`: one series, days 1, 2, 3, …
with no gaps. Anything else is rejected.

In [ ]:
#| export
def load_observations(path: Path) -> tuple[Observation, ...]:
    """Load a single, contiguous series from a strict CSV fixture."""
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ["series_id", "day", "value"]:
            raise ValueError("fixture columns must be series_id,day,value")
        rows = tuple(reader)
    if not rows:
        raise ValueError("fixture must contain at least one observation")

    observations: list[Observation] = []
    for row in rows:
        try:
            series_id = row["series_id"].strip()
            day = int(row["day"])
            value = float(row["value"])
        except (KeyError, TypeError, ValueError) as exc:
            raise ValueError("fixture contains an invalid observation") from exc
        if not series_id or not isfinite(value):
            raise ValueError("fixture contains an invalid observation")
        observations.append(Observation(series_id, day, value))

    series_ids = {item.series_id for item in observations}
    days = [item.day for item in observations]
    if len(series_ids) != 1 or days != list(range(1, len(days) + 1)):
        raise ValueError(
            "fixture must be one series with contiguous days starting at 1"
        )
    return tuple(observations)

Split at the forecast origin: days up to and including the origin are history, later
days are what the forecast is judged against.

In [ ]:
#| export
def split_at_origin(
    observations: Sequence[Observation], *, origin_day: int
) -> tuple[tuple[Observation, ...], tuple[Observation, ...]]:
    """Split history (day <= origin) from the target period (day > origin)."""
    if origin_day < 1:
        raise ValueError("origin_day must be positive")
    train = tuple(item for item in observations if item.day <= origin_day)
    target = tuple(item for item in observations if item.day > origin_day)
    if not train or not target or train[-1].day != origin_day:
        raise ValueError("origin_day must split observed history from a future target")
    return train, target

Seasonal naive: the forecast for `horizon` days ahead repeats the matching day of the
last full season. With a 7-day season, every Saturday ahead gets last Saturday's value.

In [ ]:
#| export
def seasonal_naive(
    history: Sequence[Observation], *, period: int, horizon: int
) -> float:
    """Repeat the matching day of the last full season, ``horizon`` days ahead."""
    if period <= 0:
        raise ValueError("period must be positive")
    if horizon < 1:
        raise ValueError("horizon must be at least 1")
    if len(history) < period:
        raise ValueError("history must contain one full period")
    return history[-period + (horizon - 1) % period].value

## Example

The sample fixture has 8 days. Split at day 7, then forecast day 8 from day 1. Eight
days ahead is the same weekday again, so it gets the same forecast.

In [ ]:
observations = load_observations(Path("data/synthetic_weekly_01.csv"))
history, target = split_at_origin(observations, origin_day=7)
forecast = seasonal_naive(history, period=7, horizon=1)
print(
    f"history: {len(history)} days, target day 8 = {target[0].value}, forecast = {forecast}"
)
assert len(history) == 7 and target[0].value == 13.0 and forecast == 10.0
assert seasonal_naive(history, period=7, horizon=8) == forecast